# LightGBM Model for Delinquency Prediction

Uses the **top 50 selected features** from the comprehensive feature selection pipeline.

- Train / Validation / Test split (60/20/20)
- LightGBM gradient boosting classifier
- Handles class imbalance with `scale_pos_weight`
- Records training time, scoring time, and model performance

In [27]:
import sys
import pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))

import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, roc_auc_score, confusion_matrix,
    precision_recall_curve, average_precision_score
)
import matplotlib.pyplot as plt
import time
import warnings
import importlib
warnings.filterwarnings('ignore')

import scripts.feature_selection as _fs
importlib.reload(_fs)
import scripts.model_data as _md
importlib.reload(_md)
from scripts.model_data import load_and_split, save_sorted_features


## 1. Load Data & Selected Features

In [28]:
# ── Configuration ────────────────────────────────────────────────
N_FEATURES = 50  # ← change this to use a different number of top-ranked features
FEATURES_FILE = "features_consensus_ordered.csv"

## 2. Prepare Data & Train/Val/Test Split

In [29]:
# Load features and perform 60/20/20 stratified split
X_train, X_val, X_test, y_train, y_val, y_test, available_features, features_df = \
    load_and_split(n_features=N_FEATURES, features_filename=FEATURES_FILE)

Features  : 50 (top-50)
Samples   : 10317  |  DQ rate: 8.86%
Train     : 6190  (8.85% positive)
Val       : 2063  (8.87% positive)
Test      : 2064  (8.87% positive)


## 3. Train LightGBM Model

In [30]:
# Class imbalance weight
n_neg = np.sum(y_train == 0)
n_pos = np.sum(y_train == 1)
scale_pos_weight = n_neg / n_pos
print(f'scale_pos_weight: {scale_pos_weight:.2f}')

# Create LightGBM datasets
train_data = lgb.Dataset(X_train, label=y_train, feature_name=available_features)
val_data = lgb.Dataset(X_val, label=y_val, feature_name=available_features, reference=train_data)

# LightGBM parameters
params = {
    'objective': 'binary',
    'metric': ['auc', 'binary_logloss'],
    'boosting_type': 'gbdt',
    'scale_pos_weight': scale_pos_weight,
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': -1,
    'min_child_samples': 20,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'verbose': -1,
    'seed': 42,
}

# Train with early stopping
print('\nTraining LightGBM...')
training_start_time = time.time()

evals_result = {}

callbacks = [
    lgb.early_stopping(stopping_rounds=50, verbose=True),
    lgb.log_evaluation(period=50),
    lgb.record_evaluation(evals_result),
]

model = lgb.train(
    params,
    train_data,
    num_boost_round=1000,
    valid_sets=[train_data, val_data],
    valid_names=['train', 'val'],
    callbacks=callbacks,
)

training_end_time = time.time()
training_time = training_end_time - training_start_time

print(f'\nTraining completed in {training_time:.2f}s')
print(f'Best iteration: {model.best_iteration}')
print(f'Best val AUC: {model.best_score["val"]["auc"]:.4f}')


scale_pos_weight: 10.30

Training LightGBM...
Training until validation scores don't improve for 50 rounds
[50]	train's auc: 0.973933	train's binary_logloss: 0.287323	val's auc: 0.799243	val's binary_logloss: 0.371752
Early stopping, best iteration is:
[3]	train's auc: 0.879882	train's binary_logloss: 0.269086	val's auc: 0.75882	val's binary_logloss: 0.285542

Training completed in 0.56s
Best iteration: 3
Best val AUC: 0.7588


## 5. Evaluation on All Splits

In [31]:
def evaluate(model, X, y_true, split_name):
    """Evaluate model and print metrics."""
    scoring_start = time.time()
    
    # Predict probabilities
    y_proba = model.predict(X, num_iteration=model.best_iteration)
    
    scoring_time = time.time() - scoring_start
    
    # AUC
    auc = roc_auc_score(y_true, y_proba)
    ap = average_precision_score(y_true, y_proba)
    
    # Binary predictions at 0.5 threshold
    y_pred = (y_proba >= 0.5).astype(int)
    
    print(f'\n{"="*50}')
    print(f'{split_name} Results')
    print(f'{"="*50}')
    print(f'ROC-AUC: {auc:.4f}')
    print(f'Average Precision: {ap:.4f}')
    print(f'Scoring Time: {scoring_time:.4f} seconds')
    print(f'\nClassification Report:')
    print(classification_report(y_true, y_pred, target_names=['No DQ', 'DQ']))
    print(f'Confusion Matrix:')
    print(confusion_matrix(y_true, y_pred))
    
    return y_proba, auc, ap, scoring_time

train_preds, train_auc, train_ap, train_time = evaluate(model, X_train, y_train, 'TRAIN')
val_preds, val_auc, val_ap, val_time = evaluate(model, X_val, y_val, 'VALIDATION')
test_preds, test_auc, test_ap, test_time = evaluate(model, X_test, y_test, 'TEST')


TRAIN Results
ROC-AUC: 0.8799
Average Precision: 0.3222
Scoring Time: 0.0004 seconds

Classification Report:
              precision    recall  f1-score   support

       No DQ       0.91      1.00      0.95      5642
          DQ       0.00      0.00      0.00       548

    accuracy                           0.91      6190
   macro avg       0.46      0.50      0.48      6190
weighted avg       0.83      0.91      0.87      6190

Confusion Matrix:
[[5642    0]
 [ 548    0]]

VALIDATION Results
ROC-AUC: 0.7588
Average Precision: 0.2134
Scoring Time: 0.0000 seconds

Classification Report:
              precision    recall  f1-score   support

       No DQ       0.91      1.00      0.95      1880
          DQ       0.00      0.00      0.00       183

    accuracy                           0.91      2063
   macro avg       0.46      0.50      0.48      2063
weighted avg       0.83      0.91      0.87      2063

Confusion Matrix:
[[1880    0]
 [ 183    0]]

TEST Results
ROC-AUC: 0.7658
A

## 7. Summary

In [32]:
print('LightGBM Model Summary')
print('=' * 50)
print(f'Boosting type: GBDT')
print(f'Num leaves: {params["num_leaves"]}')
print(f'Learning rate: {params["learning_rate"]}')
print(f'Best iteration: {model.best_iteration}')
print(f'Features used: {len(available_features)}')
print(f'')
print(f'Training Time: {training_time:.2f}s ({training_time/60:.2f} min)')
print(f'')
print(f'{"Split":<12} {"ROC-AUC":<12} {"Avg Precision":<16} {"Scoring Time":<14}')
print(f'{"-"*54}')
print(f'{"Train":<12} {train_auc:<12.4f} {train_ap:<16.4f} {train_time:<14.4f}s')
print(f'{"Val":<12} {val_auc:<12.4f} {val_ap:<16.4f} {val_time:<14.4f}s')
print(f'{"Test":<12} {test_auc:<12.4f} {test_ap:<16.4f} {test_time:<14.4f}s')

LightGBM Model Summary
Boosting type: GBDT
Num leaves: 31
Learning rate: 0.05
Best iteration: 3
Features used: 50

Training Time: 0.56s (0.01 min)

Split        ROC-AUC      Avg Precision    Scoring Time  
------------------------------------------------------
Train        0.8799       0.3222           0.0004        s
Val          0.7588       0.2134           0.0000        s
Test         0.7658       0.2340           0.0000        s


---
# LightGBM – All Features

Same setup but using **every available feature** instead of the top-50 selection.

In [33]:
# ── Use ALL features ─────────────────────────────────────────────
X_train_all, X_val_all, X_test_all, y_train_all, y_val_all, y_test_all, all_feature_cols, _ = \
    load_and_split(use_all=True)

Features  : 236 (all)
Samples   : 10317  |  DQ rate: 8.86%
Train     : 6190  (8.85% positive)
Val       : 2063  (8.87% positive)
Test      : 2064  (8.87% positive)


In [34]:
# ── Train LightGBM (all features) ────────────────────────────────
n_neg_all = np.sum(y_train_all == 0)
n_pos_all = np.sum(y_train_all == 1)
scale_pos_weight_all = n_neg_all / n_pos_all
print(f'scale_pos_weight: {scale_pos_weight_all:.2f}')

train_data_all = lgb.Dataset(X_train_all, label=y_train_all, feature_name=all_feature_cols)
val_data_all   = lgb.Dataset(X_val_all,   label=y_val_all,   feature_name=all_feature_cols,
                              reference=train_data_all)

params_all = {
    'objective': 'binary',
    'metric': ['auc', 'binary_logloss'],
    'boosting_type': 'gbdt',
    'scale_pos_weight': scale_pos_weight_all,
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': -1,
    'min_child_samples': 20,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'verbose': -1,
    'seed': 42,
}

print('\nTraining LightGBM (all features)...')
train_start_all = time.time()

evals_result_all = {}

model_all = lgb.train(
    params_all,
    train_data_all,
    num_boost_round=1000,
    valid_sets=[train_data_all, val_data_all],
    valid_names=['train', 'val'],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=True),
        lgb.log_evaluation(period=50),
        lgb.record_evaluation(evals_result_all),
    ],
)

training_time_all = time.time() - train_start_all
print(f'\nTraining completed in {training_time_all:.2f}s')
print(f'Best iteration: {model_all.best_iteration}')
print(f'Best val AUC: {model_all.best_score["val"]["auc"]:.4f}')


scale_pos_weight: 10.30

Training LightGBM (all features)...
Training until validation scores don't improve for 50 rounds
[50]	train's auc: 0.982883	train's binary_logloss: 0.261246	val's auc: 0.799573	val's binary_logloss: 0.356095
Early stopping, best iteration is:
[3]	train's auc: 0.895684	train's binary_logloss: 0.266388	val's auc: 0.765165	val's binary_logloss: 0.28282

Training completed in 0.65s
Best iteration: 3
Best val AUC: 0.7652


In [35]:
# ── Evaluation (all features) ────────────────────────────────────
train_preds_all, train_auc_all, train_ap_all, train_time_all = evaluate(
    model_all, X_train_all, y_train_all, 'TRAIN (all features)')
val_preds_all, val_auc_all, val_ap_all, val_time_all = evaluate(
    model_all, X_val_all, y_val_all, 'VALIDATION (all features)')
test_preds_all, test_auc_all, test_ap_all, test_time_all = evaluate(
    model_all, X_test_all, y_test_all, 'TEST (all features)')


TRAIN (all features) Results
ROC-AUC: 0.8957
Average Precision: 0.3512
Scoring Time: 0.0020 seconds

Classification Report:
              precision    recall  f1-score   support

       No DQ       0.91      1.00      0.95      5642
          DQ       0.00      0.00      0.00       548

    accuracy                           0.91      6190
   macro avg       0.46      0.50      0.48      6190
weighted avg       0.83      0.91      0.87      6190

Confusion Matrix:
[[5642    0]
 [ 548    0]]

VALIDATION (all features) Results
ROC-AUC: 0.7652
Average Precision: 0.2201
Scoring Time: 0.0000 seconds

Classification Report:
              precision    recall  f1-score   support

       No DQ       0.91      1.00      0.95      1880
          DQ       0.00      0.00      0.00       183

    accuracy                           0.91      2063
   macro avg       0.46      0.50      0.48      2063
weighted avg       0.83      0.91      0.87      2063

Confusion Matrix:
[[1880    0]
 [ 183    0]]



In [36]:
# ── Summary: All Features vs Top-50 ──────────────────────────────
print('Comparison: Top-50 Features vs All Features')
print('=' * 70)
print(f'{"Metric":<22} {"Top-50":<22} {"All Features":<22}')
print(f'{"-"*66}')
print(f'{"Num features":<22} {len(available_features):<22} {len(all_feature_cols):<22}')
print(f'{"Best iteration":<22} {model.best_iteration:<22} {model_all.best_iteration:<22}')
print(f'{"Training time (s)":<22} {training_time:<22.2f} {training_time_all:<22.2f}')
print()
print(f'{"Split":<8} {"Metric":<16} {"Top-50":<14} {"All Features":<14}')
print(f'{"-"*52}')
for split, (auc50, ap50, t50, auca, apa, ta) in {
    'Train': (train_auc, train_ap, train_time, train_auc_all, train_ap_all, train_time_all),
    'Val':   (val_auc,   val_ap,   val_time,   val_auc_all,   val_ap_all,   val_time_all),
    'Test':  (test_auc,  test_ap,  test_time,  test_auc_all,  test_ap_all,  test_time_all),
}.items():
    print(f'{split:<8} {"ROC-AUC":<16} {auc50:<14.4f} {auca:<14.4f}')
    print(f'{"":8} {"Avg Precision":<16} {ap50:<14.4f} {apa:<14.4f}')
    print(f'{"":8} {"Score time (s)":<16} {t50:<14.4f} {ta:<14.4f}')

Comparison: Top-50 Features vs All Features
Metric                 Top-50                 All Features          
------------------------------------------------------------------
Num features           50                     236                   
Best iteration         3                      3                     
Training time (s)      0.56                   0.65                  

Split    Metric           Top-50         All Features  
----------------------------------------------------
Train    ROC-AUC          0.8799         0.8957        
         Avg Precision    0.3222         0.3512        
         Score time (s)   0.0004         0.0020        
Val      ROC-AUC          0.7588         0.7652        
         Avg Precision    0.2134         0.2201        
         Score time (s)   0.0000         0.0000        
Test     ROC-AUC          0.7658         0.7551        
         Avg Precision    0.2340         0.2350        
         Score time (s)   0.0000         0.0000        

---
## 9. Regularized Model (Overfitting Fix)

Early overfitting is caused by trees that are too complex. Key changes vs the baseline:

| Parameter | Baseline | Regularized | Why |
|---|---|---|---|
| `num_leaves` | 31 | **15** | Fewer leaves → simpler trees |
| `max_depth` | -1 (unlimited) | **5** | Hard cap on tree depth |
| `min_child_samples` | 20 | **100** | Each leaf needs more data |
| `subsample` | 0.8 | **0.6** | More randomness per tree |
| `colsample_bytree` | 0.8 | **0.6** | Sample fewer features per tree |
| `reg_alpha` | 0.1 | **1.0** | L1 sparsity regularization |
| `reg_lambda` | 0.1 | **5.0** | L2 weight regularization |
| `min_split_gain` | 0 | **0.05** | Require meaningful splits only |
| `learning_rate` | 0.05 | **0.02** | Slower learning, more rounds |

In [37]:
# ── Regularized LightGBM (Top-50 features) ───────────────────────
params_reg = {
    'objective':          'binary',
    'metric':             ['auc', 'binary_logloss'],
    'boosting_type':      'gbdt',
    'scale_pos_weight':   scale_pos_weight,

    # ── tree complexity (main overfitting levers) ──
    'num_leaves':         15,     # was 31
    'max_depth':          5,      # was -1 (unlimited)
    'min_child_samples':  100,    # was 20

    # ── stochasticity ──
    'subsample':          0.6,    # was 0.8
    'colsample_bytree':   0.6,    # was 0.8

    # ── regularization ──
    'reg_alpha':          1.0,    # was 0.1
    'reg_lambda':         5.0,    # was 0.1
    'min_split_gain':     0.05,   # was 0 (no threshold)

    # ── learning ──
    'learning_rate':      0.02,   # was 0.05 (slower + more rounds)
    'verbose':            -1,
    'seed':               42,
}

evals_result_reg = {}

print('Training regularized LightGBM...')
t0_reg = time.time()

model_reg = lgb.train(
    params_reg,
    train_data,                     # same split as baseline
    num_boost_round=2000,           # more rounds at lower LR
    valid_sets=[train_data, val_data],
    valid_names=['train', 'val'],
    callbacks=[
        lgb.early_stopping(stopping_rounds=100, verbose=True),
        lgb.log_evaluation(period=100),
        lgb.record_evaluation(evals_result_reg),
    ],
)

training_time_reg = time.time() - t0_reg
print(f'\nDone in {training_time_reg:.2f}s  |  best iteration: {model_reg.best_iteration}')
print(f'Best val AUC: {model_reg.best_score["val"]["auc"]:.4f}')


Training regularized LightGBM...
Training until validation scores don't improve for 100 rounds
[100]	train's auc: 0.897783	train's binary_logloss: 0.393257	val's auc: 0.796156	val's binary_logloss: 0.427598
Early stopping, best iteration is:
[7]	train's auc: 0.84108	train's binary_logloss: 0.281743	val's auc: 0.773657	val's binary_logloss: 0.28778

Done in 0.86s  |  best iteration: 7
Best val AUC: 0.7737


In [38]:
# ── Evaluate regularized model ────────────────────────────────────
train_preds_reg, train_auc_reg, train_ap_reg, train_time_reg = evaluate(
    model_reg, X_train, y_train, 'TRAIN (regularized)')
val_preds_reg, val_auc_reg, val_ap_reg, val_time_reg = evaluate(
    model_reg, X_val, y_val, 'VALIDATION (regularized)')
test_preds_reg, test_auc_reg, test_ap_reg, test_time_reg = evaluate(
    model_reg, X_test, y_test, 'TEST (regularized)')

print('\nBaseline vs Regularized (Top-50)')
print('=' * 58)
print(f'{"Split":<8} {"Metric":<18} {"Baseline":<14} {"Regularized":<14}')
print(f'{"-"*58}')
for split_name, (auc_b, auc_r) in {
    'Train': (train_auc,     train_auc_reg),
    'Val':   (val_auc,       val_auc_reg),
    'Test':  (test_auc,      test_auc_reg),
}.items():
    print(f'{split_name:<8} {"ROC-AUC":<18} {auc_b:<14.4f} {auc_r:<14.4f}')



TRAIN (regularized) Results
ROC-AUC: 0.8411
Average Precision: 0.2907
Scoring Time: 0.0020 seconds

Classification Report:
              precision    recall  f1-score   support

       No DQ       0.91      1.00      0.95      5642
          DQ       0.00      0.00      0.00       548

    accuracy                           0.91      6190
   macro avg       0.46      0.50      0.48      6190
weighted avg       0.83      0.91      0.87      6190

Confusion Matrix:
[[5642    0]
 [ 548    0]]

VALIDATION (regularized) Results
ROC-AUC: 0.7737
Average Precision: 0.2163
Scoring Time: 0.0000 seconds

Classification Report:
              precision    recall  f1-score   support

       No DQ       0.91      1.00      0.95      1880
          DQ       0.00      0.00      0.00       183

    accuracy                           0.91      2063
   macro avg       0.46      0.50      0.48      2063
weighted avg       0.83      0.91      0.87      2063

Confusion Matrix:
[[1880    0]
 [ 183    0]]

TE

---
# LightGBM – ADASYN vs scale_pos_weight (Top-50 Features)

Compare training with **ADASYN oversampling** against the existing **scale_pos_weight** approach, both using the same top-50 features.

In [39]:
from imblearn.over_sampling import ADASYN

# Apply ADASYN to the top-50 training set
_adasyn = ADASYN(sampling_strategy='minority', random_state=42, n_neighbors=5)
X_train_ada, y_train_ada = _adasyn.fit_resample(X_train, y_train)

print(f'Before ADASYN: {np.bincount(y_train)}')
print(f'After  ADASYN: {np.bincount(y_train_ada)}')

Before ADASYN: [5642  548]
After  ADASYN: [5642 5730]


In [40]:
# ── Train LightGBM (ADASYN, no scale_pos_weight) ─────────────────
train_data_ada = lgb.Dataset(X_train_ada, label=y_train_ada, feature_name=available_features)
val_data_ada   = lgb.Dataset(X_val, label=y_val, feature_name=available_features,
                              reference=train_data_ada)

params_ada = {
    'objective': 'binary',
    'metric': ['auc', 'binary_logloss'],
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': -1,
    'min_child_samples': 20,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'verbose': -1,
    'seed': 42,
}

print('Training LightGBM (ADASYN)...')
_t0_ada = time.time()
evals_result_ada = {}

model_ada = lgb.train(
    params_ada,
    train_data_ada,
    num_boost_round=1000,
    valid_sets=[train_data_ada, val_data_ada],
    valid_names=['train', 'val'],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=True),
        lgb.log_evaluation(period=50),
        lgb.record_evaluation(evals_result_ada),
    ],
)

training_time_ada = time.time() - _t0_ada
print(f'\nTraining completed in {training_time_ada:.2f}s')
print(f'Best iteration: {model_ada.best_iteration}')
print(f'Best val AUC: {model_ada.best_score["val"]["auc"]:.4f}')

Training LightGBM (ADASYN)...
Training until validation scores don't improve for 50 rounds
[50]	train's auc: 0.982799	train's binary_logloss: 0.246631	val's auc: 0.779735	val's binary_logloss: 0.344223
[100]	train's auc: 0.993038	train's binary_logloss: 0.15002	val's auc: 0.797156	val's binary_logloss: 0.287719
[150]	train's auc: 0.997048	train's binary_logloss: 0.108067	val's auc: 0.799068	val's binary_logloss: 0.276561
Early stopping, best iteration is:
[145]	train's auc: 0.996775	train's binary_logloss: 0.111365	val's auc: 0.799696	val's binary_logloss: 0.276871

Training completed in 2.18s
Best iteration: 145
Best val AUC: 0.7997


In [41]:
# ── Evaluate ADASYN model on ORIGINAL (non-oversampled) splits ────
train_preds_ada, train_auc_ada, train_ap_ada, train_time_ada = evaluate(
    model_ada, X_train, y_train, 'TRAIN (ADASYN)')
val_preds_ada, val_auc_ada, val_ap_ada, val_time_ada = evaluate(
    model_ada, X_val, y_val, 'VALIDATION (ADASYN)')
test_preds_ada, test_auc_ada, test_ap_ada, test_time_ada = evaluate(
    model_ada, X_test, y_test, 'TEST (ADASYN)')


TRAIN (ADASYN) Results
ROC-AUC: 0.9698
Average Precision: 0.7531
Scoring Time: 0.0084 seconds

Classification Report:
              precision    recall  f1-score   support

       No DQ       0.97      0.98      0.97      5642
          DQ       0.75      0.64      0.69       548

    accuracy                           0.95      6190
   macro avg       0.86      0.81      0.83      6190
weighted avg       0.95      0.95      0.95      6190

Confusion Matrix:
[[5526  116]
 [ 200  348]]

VALIDATION (ADASYN) Results
ROC-AUC: 0.7997
Average Precision: 0.2395
Scoring Time: 0.0024 seconds

Classification Report:
              precision    recall  f1-score   support

       No DQ       0.93      0.95      0.94      1880
          DQ       0.29      0.22      0.25       183

    accuracy                           0.88      2063
   macro avg       0.61      0.59      0.59      2063
weighted avg       0.87      0.88      0.88      2063

Confusion Matrix:
[[1779  101]
 [ 142   41]]

TEST (ADASYN

In [42]:
# ── Side-by-side comparison ───────────────────────────────────────
print('scale_pos_weight  vs  ADASYN  (Top-50 features)')
print('=' * 52)
print(f'{"Split":<8} {"scale_pos_weight":<20} {"ADASYN":<10}')
print('-' * 52)
for _split, _spw_auc, _ada_auc in [
    ('Train', train_auc,  train_auc_ada),
    ('Val',   val_auc,    val_auc_ada),
    ('Test',  test_auc,   test_auc_ada),
]:
    print(f'{_split:<8} {_spw_auc:<20.4f} {_ada_auc:<10.4f}')

scale_pos_weight  vs  ADASYN  (Top-50 features)
Split    scale_pos_weight     ADASYN    
----------------------------------------------------
Train    0.8799               0.9698    
Val      0.7588               0.7997    
Test     0.7658               0.7841    


---
# LightGBM – Filtered vs Original Features (ADASYN, Top-50)

Train on the scoring-exclusion-filtered population (`filtered_features_consensus_ordered.csv`) and compare ROC-AUC against the original full population.

In [43]:
# ── Load filtered features ────────────────────────────────────────
X_train_flt, X_val_flt, X_test_flt, y_train_flt, y_val_flt, y_test_flt, feats_flt, _ = \
    load_and_split(n_features=N_FEATURES, features_filename="filtered_features_consensus_ordered.csv")

# Apply ADASYN
_adasyn_flt = ADASYN(sampling_strategy='minority', random_state=42, n_neighbors=5)
X_train_flt_ada, y_train_flt_ada = _adasyn_flt.fit_resample(X_train_flt, y_train_flt)
print(f'Filtered – Before ADASYN: {np.bincount(y_train_flt)}')
print(f'Filtered – After  ADASYN: {np.bincount(y_train_flt_ada)}')

Features  : 50 (top-50)
Samples   : 9090  |  DQ rate: 8.70%
Train     : 5454  (8.71% positive)
Val       : 1818  (8.69% positive)
Test      : 1818  (8.69% positive)
Filtered – Before ADASYN: [4979  475]
Filtered – After  ADASYN: [4979 4872]


In [44]:
# ── Train LightGBM on filtered features (ADASYN) ─────────────────
_td_flt = lgb.Dataset(X_train_flt_ada, label=y_train_flt_ada, feature_name=feats_flt)
_vd_flt = lgb.Dataset(X_val_flt, label=y_val_flt, feature_name=feats_flt, reference=_td_flt)

_params_flt = {**params_ada}  # same hyperparameters

print('Training LightGBM (filtered, ADASYN)...')
_t0_flt = time.time()
_er_flt = {}

model_flt = lgb.train(
    _params_flt, _td_flt,
    num_boost_round=1000,
    valid_sets=[_td_flt, _vd_flt],
    valid_names=['train', 'val'],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=True),
        lgb.log_evaluation(period=50),
        lgb.record_evaluation(_er_flt),
    ],
)

training_time_flt = time.time() - _t0_flt
print(f'\nTraining completed in {training_time_flt:.2f}s')
print(f'Best iteration: {model_flt.best_iteration}')
print(f'Best val AUC: {model_flt.best_score["val"]["auc"]:.4f}')

Training LightGBM (filtered, ADASYN)...
Training until validation scores don't improve for 50 rounds
[50]	train's auc: 0.987553	train's binary_logloss: 0.225146	val's auc: 0.766902	val's binary_logloss: 0.327105
[100]	train's auc: 0.995979	train's binary_logloss: 0.131618	val's auc: 0.773383	val's binary_logloss: 0.283121
[150]	train's auc: 0.999041	train's binary_logloss: 0.0870787	val's auc: 0.776197	val's binary_logloss: 0.272411
[200]	train's auc: 0.999861	train's binary_logloss: 0.0617543	val's auc: 0.777833	val's binary_logloss: 0.271557
Early stopping, best iteration is:
[174]	train's auc: 0.999584	train's binary_logloss: 0.0735186	val's auc: 0.777669	val's binary_logloss: 0.271062

Training completed in 2.31s
Best iteration: 174
Best val AUC: 0.7777


In [45]:
# ── Evaluate filtered model ───────────────────────────────────────
_, train_auc_flt, _, _ = evaluate(model_flt, X_train_flt, y_train_flt, 'TRAIN (filtered, ADASYN)')
_, val_auc_flt,   _, _ = evaluate(model_flt, X_val_flt,   y_val_flt,   'VALIDATION (filtered, ADASYN)')
_, test_auc_flt,  _, _ = evaluate(model_flt, X_test_flt,  y_test_flt,  'TEST (filtered, ADASYN)')

# ── Final comparison ──────────────────────────────────────────────
print('\nOriginal (scale_pos_weight)  vs  Original (ADASYN)  vs  Filtered (ADASYN)  — Top-50')
print('=' * 76)
print(f'{"Split":<8} {"Original (spw)":<20} {"Original (ADASYN)":<22} {"Filtered (ADASYN)":<20}')
print('-' * 76)
for _split, _spw, _ada, _flt in [
    ('Train', train_auc,     train_auc_ada, train_auc_flt),
    ('Val',   val_auc,       val_auc_ada,   val_auc_flt),
    ('Test',  test_auc,      test_auc_ada,  test_auc_flt),
]:
    print(f'{_split:<8} {_spw:<20.4f} {_ada:<22.4f} {_flt:<20.4f}')


TRAIN (filtered, ADASYN) Results
ROC-AUC: 0.9957
Average Precision: 0.9643
Scoring Time: 0.0079 seconds

Classification Report:
              precision    recall  f1-score   support

       No DQ       0.98      1.00      0.99      4979
          DQ       0.97      0.78      0.86       475

    accuracy                           0.98      5454
   macro avg       0.97      0.89      0.93      5454
weighted avg       0.98      0.98      0.98      5454

Confusion Matrix:
[[4966   13]
 [ 104  371]]

VALIDATION (filtered, ADASYN) Results
ROC-AUC: 0.7777
Average Precision: 0.2511
Scoring Time: 0.0010 seconds

Classification Report:
              precision    recall  f1-score   support

       No DQ       0.93      0.96      0.95      1660
          DQ       0.36      0.22      0.27       158

    accuracy                           0.90      1818
   macro avg       0.64      0.59      0.61      1818
weighted avg       0.88      0.90      0.89      1818

Confusion Matrix:
[[1600   60]
 [ 124 